# RL post-training: from an LLM judge to RLAIF and RLHF

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/somilsin/Machine-Learning/blob/main/Transformers_Large-Language-Models/3_RL_Post_Training.ipynb)

This notebook runs the two projects in [`rl-post-training/`](rl-post-training/README.md) end to end:

* **Part A · RLAIF.** The Yoda-style judge from [2_LLM_Finetuning.ipynb](2_LLM_Finetuning.ipynb) becomes a reward model. It labels preference pairs for DPO (offline) and scores rollouts live for GRPO (online).
* **Part B · RLHF.** Supervised fine-tuning on human-preferred replies, a Bradley-Terry reward model from human preferences and GRPO against it. Then the same RL loop with a verifiable reward: a GSM8K answer checker.

Every training loop is plain PyTorch in [`rl-post-training/rlkit.py`](rl-post-training/rlkit.py). The notebook prints each key function next to the run that uses it.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. `QUICK = True` gives a first pass with small settings. `QUICK = False` uses the README defaults. Everything in `runs/` is lost when the Colab session ends (an optional cell at the end copies the logs to Drive).

**Status:** the scripts were tested on a local RTX 3050 Ti (4 GB) with tiny settings. This notebook has not yet been executed end to end on a T4. Its cells carry no saved outputs.

## 0 · Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
if not os.path.exists("/content/Machine-Learning"):
    !git clone --depth 1 https://github.com/somilsin/Machine-Learning.git /content/Machine-Learning
%cd /content/Machine-Learning/Transformers_Large-Language-Models/rl-post-training
!pip install -q -r requirements.txt
!pip install -q mitdeeplearning --no-deps --no-build-isolation

`QUICK` controls every run size below. On a T4 the policy trains in fp32 (the T4 has no bf16 and fp16 training without a loss scaler can underflow) while the frozen judge and reward model run in fp16.

In [ ]:
QUICK = True

# Part A · Yoda RLAIF
A_SFT, A_PREFS, A_DPO_EPOCHS, A_GRPO, A_EVAL = (150, 80, 2, 40, 30) if QUICK else (300, 300, 2, 150, 40)
# Part B · hh-rlhf RLHF and GSM8K
B_SFT, B_SFT_N, B_RM_N, B_RM_TEST = (300, 3000, 1500, 300) if QUICK else (600, 6000, 4000, 400)
B_GRPO, B_EVAL, B_GSM, B_GSM_EVAL = (40, 50, 40, 100) if QUICK else (200, 100, 200, 200)

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["DATASETS_VERBOSITY"] = "error"

In [ ]:
import importlib.util, inspect, json, sys
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, ".")
import rlkit

def load(name, path):
    """Import a project's common.py under its own name (both projects have one)."""
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

def show(fn):
    print(inspect.getsource(fn))

def plot(path, keys, title):
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    fig, axes = plt.subplots(1, len(keys), figsize=(3.6 * len(keys), 2.8), squeeze=False)
    for ax, k in zip(axes[0], keys):
        pts = [(r["step"], r[k]) for r in rows if k in r]
        ax.plot([p[0] for p in pts], [p[1] for p in pts])
        ax.set_title(k)
        ax.set_xlabel("step")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

yc = load("yoda_common", "rlaif_yoda/common.py")
hc = load("hh_common", "rlhf_hh/common.py")
print("device:", rlkit.DEVICE, "| train dtype:", rlkit.TRAIN_DTYPE, "| frozen-model dtype:", rlkit.DTYPE)

# Part A · From an LLM judge to RLAIF

RLHF needs a reward signal from human preference labels. RLAIF replaces the humans with a model that follows a written rubric (Constitutional AI calls the rubric a constitution). The earlier notebook already had every piece:

| In 2_LLM_Finetuning.ipynb | In RLAIF |
|---|---|
| The judge's system prompt with a 0-10 Yoda score | The rubric that replaces human labels |
| `LLMJudgeEvaluator.score()` returning 0-1 | The reward model r(x, y) |
| Plain Dolly answers and Yoda rewrites as controls | Reward-model validation before any training |
| The note that the judge can reward superficial phrasing | Reward hacking, handled here with a second rubric |
| Scoring samples after training | Scoring samples during training and updating on them |

The pipeline: **SFT → check the judge → judge-labelled pairs → DPO** and **SFT → judge in the loop → GRPO** then one evaluation over every stage.

## A0 · Supervised fine-tuning

The same data and split as the earlier notebook: Dolly instructions paired with the course's Yoda rewrites, 80/20 with seed 7. The loss covers only answer tokens. Two functions do the work. `pad_batch` builds a mask that marks which positions predict an answer token. `token_logprobs` returns log π(next token) at every position.

In [ ]:
show(rlkit.pad_batch)
show(rlkit.token_logprobs)

Answers longer than the 256-token budget are dropped rather than cut. A cut answer loses its end-of-turn token and teaches the model never to stop, which later breaks RL (every rollout runs to the token cap).

In [ ]:
!python rlaif_yoda/0_sft.py --steps {A_SFT}

In [ ]:
plot("rlaif_yoda/runs/logs/sft.jsonl", ["loss"], "A0 · SFT loss (answer tokens only)")

## A1 · Turn the judge into a reward and check it

Two changes from the earlier notebook's judge:

1. **Expected-digit scoring.** The judge answers with one digit 0-9. Instead of parsing generated text the code reads the probability of each digit at the answer position and takes the expected value. That is one forward pass, a smooth reward and no parse failures.
2. **Two rubrics.** `reward = 0.7 × style + 0.3 × answers the question`. A style-only reward can be maximised by talking like Yoda and ignoring the question.

In [ ]:
show(yc.LocalJudge._score)
print(yc.STYLE_RUBRIC)

A quick look with the earlier notebook's three test sentences. A useful judge scores the first lowest.

In [ ]:
import torch
judge = yc.LocalJudge()
tests = ["Tennis is a fun sport. But you must concentrate.",
         "Fun sport, tennis is. But work hard, you must.",
         "Hard to see, the dark side is."]
for t, s in zip(tests, judge.style(tests)):
    print(f"{s:.2f}  {t}")
del judge
torch.cuda.empty_cache()

Now the real check: score the known-plain and known-Yoda texts from the held-out split. The style gap must be at least 0.15 before the judge is used as a reward. In local tests a 0.5B judge failed this check (0.58 vs 0.62) and the default 1.5B judge passed (0.26 vs 0.46). If it fails here, rerun with `--judge_model Qwen/Qwen2.5-3B-Instruct`.

In [ ]:
!python rlaif_yoda/4_eval.py --models none --n {A_EVAL}

## A2 · Offline RLAIF: judge-labelled pairs and DPO

For each training prompt the SFT model writes 4 answers. The judge scores them and the best and worst become a (chosen, rejected) pair when their scores differ by at least 0.1.

DPO then trains directly on the pairs. Its loss comes from the KL-regularised RL objective: the optimal policy implies the reward r = β log(π/π_ref) + const. Placing that reward in the Bradley-Terry model cancels the constant. The reference π_ref is the SFT model, which here is the same weights with the LoRA switched off.

In [ ]:
show(rlkit.dpo_loss)

In [ ]:
!python rlaif_yoda/1_build_prefs.py --prompts {A_PREFS}

In [ ]:
pairs = pd.read_json("rlaif_yoda/runs/prefs.jsonl", lines=True)
print(len(pairs), "pairs")
pd.set_option("display.max_colwidth", 160)
pairs[["prompt", "chosen", "chosen_score", "rejected", "rejected_score"]].head(3)

In [ ]:
!python rlaif_yoda/2_dpo.py --epochs {A_DPO_EPOCHS}

What to look for: `reward_acc` should climb above 0.5 and `margin` should grow. If `chosen_logp` falls together with `rejected_logp` the model is lowering the probability of both answers. That is a known DPO failure mode.

In [ ]:
plot("rlaif_yoda/runs/logs/dpo.jsonl", ["loss", "reward_acc", "margin", "chosen_logp", "rejected_logp"], "A2 · DPO")

## A3 · Online RLAIF: the judge in the loop with GRPO

Each step samples 4 answers for each of 4 prompts, scores all 16 with the judge and compares every answer with the others for the same prompt. That group comparison replaces PPO's learned value network.

In [ ]:
show(rlkit.group_advantages)
show(rlkit.grpo_loss)

The full step: rollout → reward → group advantage → one clipped policy-gradient update with a k3 KL penalty to the SFT model. With one update per rollout batch the old policy equals the current one: `old_logp = logp.detach()`: the ratio is 1 in value but still carries the gradient.

In [ ]:
show(rlkit.grpo_step)

In [ ]:
!python rlaif_yoda/3_grpo.py --steps {A_GRPO}

Read the curves together. Rising `style` with falling `answer` means the policy is trading content for style. Rising `len` with rising reward can mean the judge likes longer text. `kl` starts at exactly 0 because the LoRA starts as a no-op.

In [ ]:
plot("rlaif_yoda/runs/logs/grpo.jsonl", ["reward", "style", "answer", "kl", "len", "truncated"], "A3 · GRPO with the judge as reward")

## A4 · Evaluate every stage

Same held-out prompts and the same judge for the base model, SFT, DPO and GRPO next to both controls. A model's score only means something relative to the two control rows.

In [ ]:
!python rlaif_yoda/4_eval.py --n {A_EVAL}

In [ ]:
pd.DataFrame(json.load(open("rlaif_yoda/runs/eval.json", encoding="utf-8")))[["name", "style", "style_sd", "answer", "answer_sd", "words"]].round(3)

# Part B · Classic RLHF on human preferences

Three stages as in InstructGPT: supervised fine-tuning, a reward model from human comparisons and RL against that reward model. Data: Anthropic/hh-rlhf (helpful-base). Each row holds two conversations that share every turn except the last assistant reply, one chosen by a human and one rejected.

In [ ]:
ex = hc.hh_pairs("train", 1)[0]
for m in ex["prompt"]:
    print(f"{m['role'].upper()}: {m['content'][:200]}")
print("\nCHOSEN:  ", ex["chosen"][:300])
print("REJECTED:", ex["rejected"][:300])

## B1 · SFT on human-preferred replies

Qwen2.5-0.5B (the base model, not the instruct one) learns the chat format from the chosen replies. The loss covers only the final assistant turn.

In [ ]:
!python rlhf_hh/1_sft.py --n {B_SFT_N} --steps {B_SFT}

In [ ]:
plot("rlhf_hh/runs/logs/sft.jsonl", ["loss"], "B1 · SFT loss")

## B2 · Reward model (Bradley-Terry)

The reward model is the SFT model with its language-model head swapped for a scalar head. It is trained so the chosen reply scores higher than the rejected one:

`loss = −log σ(r(chosen) − r(rejected))`

It prints held-out pairwise accuracy before and after training. Human preference data is noisy: expect well under 100%. In the local smoke test, a tiny run on 120 pairs reached 68% on 60 held-out pairs.

In [ ]:
!python rlhf_hh/2_reward_model.py --n {B_RM_N} --n_test {B_RM_TEST}

In [ ]:
plot("rlhf_hh/runs/logs/rm.jsonl", ["loss", "batch_acc", "margin"], "B2 · reward model")

## B3 · RLHF: GRPO against the reward model

The same `grpo_step` as Part A. Only the reward function changes: the reward model scores each sampled reply in its conversation.

In [ ]:
!python rlhf_hh/3_grpo.py --reward rm --steps {B_GRPO}

Reward models trained on human preferences often favour longer replies. If `len` climbs with `reward` rerun with `--len_penalty 0.05` and compare both runs on held-out prompts.

In [ ]:
plot("rlhf_hh/runs/logs/grpo_rm.jsonl", ["reward", "kl", "len", "truncated"], "B3 · GRPO against the reward model")

In [ ]:
!python rlhf_hh/4_eval.py --mode rm --n {B_EVAL}

## B4 · The same loop with a verifiable reward (GSM8K)

No reward model: a rule checks the final number. That is the RL-with-verifiable-rewards setup behind DeepSeek-R1 and the natural fit for scientific models, where units, conservation laws and solvers can check answers.

The reward is 1.0 for the right number plus 0.1 for the requested `#### <number>` format. Answer extraction accepts `####`, `\boxed{}` or the last number: Qwen2.5-Instruct writes `\boxed{}` and ignores the `####` instruction. A strict extractor gave every sample a reward of 0 in local tests. Every group was flat and the gradient was exactly zero.

In [ ]:
show(hc.extract_answer)
show(hc.gsm8k_reward)
for t, gold in [(r"so the answer is \boxed{1,234}.", "1234"), ("work... #### 42", "42"), ("#### 41", "42")]:
    print(f"{hc.gsm8k_reward(t, gold):.1f}  {t}")

In [ ]:
!python rlhf_hh/3_grpo.py --reward gsm8k --steps {B_GSM}

`flat_groups` is the share of prompts where all 8 samples got the same reward and so gave no gradient. When it sits near 1 the prompts are too easy or too hard. DAPO's dynamic sampling exists to refill those batches.

In [ ]:
plot("rlhf_hh/runs/logs/grpo_gsm8k.jsonl", ["reward", "flat_groups", "kl", "len", "truncated"], "B4 · GRPO with a GSM8K verifier")

In [ ]:
!python rlhf_hh/4_eval.py --mode gsm8k --n {B_GSM_EVAL}

# What to try next

1. **No KL:** `!python rlaif_yoda/3_grpo.py --beta 0 --steps {A_GRPO} --out rlaif_yoda/runs/yoda_grpo_nokl` and compare `kl`, `len` and the eval table.
2. **Length control:** rerun B3 with `--len_penalty 0.05` and compare win rate and length.
3. **Style only:** `--w_style 1.0` in A3, then read the samples. Does the policy stop answering the questions?
4. **A stronger judge:** `--judge_model Qwen/Qwen2.5-3B-Instruct` in A1 and A3 or `--judge openrouter` with an `OPENROUTER_API_KEY` (paid).
5. **DPO β:** `--beta 0.5` in A2 keeps the policy closer to the SFT model.

## Reading the logs

| Field | Healthy | Worry when |
|---|---|---|
| `reward` | rises then flattens | rises while held-out eval falls (reward hacking) |
| `kl` | grows slowly | jumps suddenly |
| `len` | stable or slowly rising | climbs fast with reward |
| `truncated` | low | high |
| `flat_groups` | below 0.5 | near 1.0 |
| `loss` | near 0 in value | expected: advantages sum to 0 within a group |

In [ ]:
# Optional: keep the logs and eval results after the session ends.
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p /content/drive/MyDrive/rl-post-training
# !cp -r rlaif_yoda/runs/logs rlaif_yoda/runs/eval.json rlhf_hh/runs/logs /content/drive/MyDrive/rl-post-training/